# Разбор строки подключения `tg://proxy`

Этот блокнот разбирает ссылку вида `tg://proxy?server=...&port=...&secret=...` — тот формат, который Telegram отдаёт при настройке прокси.

**Зачем**: в приложении NVR настройки уведомлений принимают адрес прокси. Формат `tg://` удобнее копировать из Telegram, поэтому его разбор добавлен прямо в интерфейс.

> **Важно про тип прокси.** Если секрет начинается с `dd`, это **fake-TLS (MTProto 2.0)**, а не SOCKS5. Проверено на живом прокси `192.168.1.1:1443`: порт открыт, но на приветствие SOCKS5, на HTTP CONNECT и на TLS-хендшейк он отвечает сбросом соединения. Это значит, что через него **нельзя** пропустить произвольный HTTPS-запрос к `api.telegram.org` — такие прокси умеют говорить только на языке MTProto.


## 1. Разбор строки подключения `tg://proxy`

Ссылка — это обычный URI: схема `tg`, хост `proxy` (это маршрут, а не адрес!), а все данные лежат в query-параметрах.

Разберём её как текст: по схеме, хосту и параметрам.


In [ ]:
# Исходная строка, скопированная из Telegram при настройке прокси.
proxy_url = 'tg://proxy?server=192.168.1.1&port=1443&secret=dd3f4f0b40da350109fc0d594aeec3c2c5'

# Показываем структуру: где схема, где хост, где параметры.
print('Строка целиком:')
print(' ', proxy_url)
print()

print('Схема  :', proxy_url.split('://')[0])
print('Хост   :', proxy_url.split('://')[1].split('?')[0])
params_part = proxy_url.split('?')[1]
print('Параметры:')
for item in params_part.split('&'):
    key, _, value = item.partition('=')
    print(f'  {key:8} = {value}')


## 2. Парсинг параметров через `urlparse`

Разбирать строку вручную (`split`) можно, но `urllib.parse` корректно обрабатывает экранирование и повторяющиеся параметры. Используем его — тот же разбор потом повторяется на Go в бэкенде.


In [ ]:
from urllib.parse import urlparse, parse_qs

parsed = urlparse(proxy_url)
params = parse_qs(parsed.query)

# parse_qs возвращает списки: один параметр может быть указан несколько раз.
server = params['server'][0]
port = int(params['port'][0])
secret = params['secret'][0]

print('Данные прокси:')
print(f'  server = {server}')
print(f'  port   = {port}  (тип {type(port).__name__})')
print(f'  secret = {secret}')
print()
print('Все параметры, как их вернул разбор:', params)


## 3. Валидация host, port и secret

Проверки нужны не для формальности: ошибка в адресе прокси проявляется как «нет связи», и без разбора причины оператор ищет её вслепую.

Проверяем три вещи: адрес — корректный IPv4, порт в допустимом диапазоне, секрет — строка из hex-символов чётной длины.


In [ ]:
import ipaddress
import re


def validate_proxy(server, port, secret):
    """Проверяет параметры прокси и возвращает (успех, пояснение).

    Пояснение нужно для интерфейса: оператор должен видеть, что именно
    не так, а не общее «неверный адрес».
    """
    # Адрес: принимаем IPv4, имя хоста тоже допустимо, но IPv4 проверяем строго.
    try:
        ipaddress.IPv4Address(server)
    except (ipaddress.AddressValueError, ValueError):
        # Не IPv4 — пробуем как имя хоста (например, proxy.example.com).
        if not re.fullmatch(r'[A-Za-z0-9.-]+', server or ''):
            return False, 'адрес прокси не похож ни на IPv4, ни на имя хоста'

    if not isinstance(port, int) or not (1 <= port <= 65535):
        return False, 'порт должен быть числом от 1 до 65535'

    if not secret:
        return False, 'секрет не задан'
    if len(secret) % 2 != 0:
        return False, 'секрет должен быть строкой из hex-символов чётной длины'
    if not re.fullmatch(r'[0-9a-fA-F]+', secret):
        return False, 'секрет содержит недопустимые символы (ожидаются только 0-9, a-f)'

    return True, 'параметры корректны'


for values in [
    (server, port, secret),
    ('192.168.1.1', 1443, 'zzz'),      # не hex
    ('192.168.1.1', 0, secret),        # порт вне диапазона
    ('', port, secret),                # пустой адрес
]:
    ok, message = validate_proxy(*values)
    mark = 'OK  ' if ok else 'ОШИБКА'
    print(f'{mark} {values[0]!r}:{values[1]} — {message}')


## 4. Формирование словаря конфигурации прокси

Соберём разобранные значения в один словарь — в таком виде их удобно передавать в клиент и показывать в интерфейсе.

Для MTProto в Python используется `ConnectionTcpMTProxyRandomizedIntermediate` из Telethon — обратите внимание, что там нужны **все три** значения: адрес, порт и секрет. Именно секрет отличает MTProto от обычного SOCKS5.


In [ ]:
proxy_config = {'server': server, 'port': port, 'secret': secret}
print('Конфигурация прокси:')
for key, value in proxy_config.items():
    print(f'  {key:8} = {value}')

# Определяем тип прокси по первому байту секрета.
# Это ключевое отличие: dd = fake-TLS, ee = обычный MTProto, и ни то, ни другое
# не является SOCKS5 — произвольный HTTPS через них не пройдёт.
first_byte = bytes.fromhex(secret)[0]
kinds = {
    0xdd: 'fake-TLS (MTProto 2.0)',
    0xee: 'MTProto в обфусцированном виде',
}
print()
print(f'Первый байт секрета: 0x{first_byte:02x}')
print('Тип прокси:', kinds.get(first_byte, 'неизвестный (возможно, обычный MTProto)'))
print()
print('Важно: это НЕ SOCKS5. Для Bot API напрямую такой прокси не подходит —')
print('он умеет передавать только трафик Telegram. Для Bot API нужен SOCKS5-вход')

# Телеграм настраивается так (требуется pip install telethon):
#
#     from telethon import TelegramClient
#     from telethon.network import ConnectionTcpMTProxyRandomizedIntermediate
#
#     client = TelegramClient(
#         'session', api_id, api_hash,
#         connection=ConnectionTcpMTProxyRandomizedIntermediate,
#         proxy=(proxy_config['server'], proxy_config['port'], proxy_config['secret']),
#     )
#
# Обратите внимание: здесь секрет передаётся как строка hex, а не как байты.
print('(пример настройки Telethon оставлен комментарием — библиотека не установлена)')


## 5. Проверка корректности секрета (hex)

Секрет — это hex-строка, которая после декодирования даёт байты ключа. Разберём её подробно: длина в байтах и первые байты показывают, что именно закодировано.


In [ ]:
def secret_info(secret):
    """Разбирает секрет и возвращает сведения о нём."""
    checks = []
    checks.append(('длина кратна 2', len(secret) % 2 == 0))
    checks.append(('только hex-символы', bool(re.fullmatch(r'[0-9a-fA-F]+', secret))))

    raw = None
    try:
        raw = bytes.fromhex(secret)
        checks.append(('декодируется из hex', True))
    except ValueError as e:
        checks.append((f'декодируется из hex ({e})', False))

    return checks, raw


checks, raw = secret_info(secret)
print(f'Секрет: {secret}')
print(f'Длина строки: {len(secret)} символов')
for name, ok in checks:
    print(f'  [{"да" if ok else "НЕТ"}] {name}')

if raw is not None:
    print()
    print(f'Длина в байтах: {len(raw)}')
    print(f'Байты: {raw.hex()}')
    print(f'Первые 2 байта: {raw[:2].hex()}')

    # Структура fake-TLS секрета: 1 байт признака (0xdd) + 16 байт ключа.
    if raw[0] == 0xdd:
        print()
        print('Это fake-TLS: 0xdd + 16 байт ключа.')
        print(f'Ключ для подключения: {raw[1:].hex()}')


## 6. Сборка обратной `tg://` ссылки из параметров

Обратная операция нужна для двух вещей: проверить, что разбор ничего не потерял, и показать оператору нормализованный вид адреса — чтобы он мог сверить его с тем, что показывает Telegram.


In [ ]:
from urllib.parse import urlencode


def build_proxy_url(server, port, secret):
    """Собирает ссылку tg://proxy из отдельных параметров."""
    query = urlencode({'server': server, 'port': port, 'secret': secret})
    return f'tg://proxy?{query}'


rebuilt = build_proxy_url(server, port, secret)
print('Исходная строка :', proxy_url)
print('Собранная заново :', rebuilt)

# Проверяем, что разбор и сборка взаимно обратны: если строки совпали,
# значит ни один параметр не потерян и не искажён.
assert rebuilt == proxy_url, 'строки не совпали — разбор потерял данные'
print()
print('Строки совпали: разбор и сборка взаимно обратны.')

# Сравниваем по параметрам — так надёжнее, если порядок параметров в ссылке
# отличается (Telegram его не гарантирует).
assert parse_qs(urlparse(rebuilt).query) == parse_qs(urlparse(proxy_url).query)
print('Параметры совпадают поключево.')

# Итоговая конфигурация для приложения.
print()
print('Итог для настроек уведомлений:')
print(f'  Прокси нужен : да (прямой доступ к api.telegram.org из России закрыт)')
print(f'  Тип          : {kinds.get(first_byte, "MTProto")}')
print(f'  Адрес и порт : {server}:{port}')
print(f'  Секрет       : {secret[:6]}... (скрыт)')


## Вывод

Строка разобрана и проверена. Ключевой результат — **тип прокси**:

Секрет начинается с `0xdd`, значит это **fake-TLS (MTProto 2.0)**. Такой прокси:

- принимает TLS-подобный трафик и передаёт его на серверы Telegram по протоколу MTProto;
- **не является SOCKS5** и не пропускает произвольные HTTPS-запросы;
- следовательно, **не подходит** для Bot API напрямую.

Проверено на этом прокси: порт `192.168.1.1:1443` открыт, но на приветствие SOCKS5, на HTTP CONNECT и на TLS-хендшейк он отвечает разрывом соединения — то есть принимает только свой протокол.

### Что делать, чтобы отправка уведомлений заработала

| Вариант | Что указать в настройках |
|---|---|
| Поставить `mtg` рядом с приложением | `socks5://127.0.0.1:1080` — заработает **без изменений кода** |
| У роутера есть SOCKS5-вход | его адрес и порт |
| Своя поддержка MTProto в приложении | сложно: нужен полный протокол MTProto |

Поле в настройках принимает адрес прокси, поэтому вариант с `mtg` — самый быстрый путь.
